<a href="https://colab.research.google.com/github/CarlosfcPinheiro/pibic-api-llm-integration/blob/main/pibic_grpc_service_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teste de requisições para LLMs utilizando padrão gRPC (Remote Procedure Call)
O objetivo principal é testar e avaliar o tempo de resposta de requisições para uma API que interaja com LLMs, utilizando o padrão gRPC (Remote Procedure Call).

## Código e estrutura
Para análise, são utilizados **8 prompts diferentes**, que buscam explorar diferentes tamanhos e temáticas de texto. Nesse script de teste, cada cliente/producer é responsável por enviar um prompt (uma request) para a fila de requests `request_summarize_queue`, sendo **30 clientes/producers** para cada prompt, onde são armazenado os tempos de resposta de todas as requests para cada prompt, respectivamente.

O arquivo .csv que contém os dados de tempo da request é lido posteriormente para compor os dados de análise, que também são escritos em um outro csv.

Além disso, cada conjunto de testes são executados utilizando **4 modelos diferentes**, que fundamentalmente são especializados em sumarização, mas com objetivos diferentes. Os modelos utilizados são:

- facebook/bart_large_cnn
- google/pegasus_cnn_dailymail
- knkarthick/MEETING_SUMMARY
- google/pegasus_xsum

O worker/consumer é o responsável por receber as mensagens (corpo das requests) da fila `request_summarize_queue`, processar, realizar a request para a API gateway de LLM, e devolver a resposta para o cliente, publicando a resposta em sua fila de respostas exclusiva.

## Padrões e bibliotecas
O padrão a ser utilizado é o de **Filas assíncronas**, dado que utiliza uma fila de mensagens (request_summarize_queue) com diferentes clientes/producers atuando em **threads únicas**.

As principais bibliotecas utilizadas são:
- time (funções de manipulação de tempo)
- requests (envio de requisições HTTP)
- csv (manipulação de arquivos csv)
- statistics (funções para cálculos estatísticos)
- numpy (funções para manipulação de arrays e matrizes)
- pika (SDK python do RabbitMQ)
- threading (gerenciamento de threads)

## Métricas analisadas
As métricas utilizadas para análise do tempo de resposta da requisição para cada prompt foram as seguintes:
- Média de tempo (tempo médio de resposta das requisições)
- Máximo e Mínimo (tempo com valores máximos e mínimos)
- Desvio padrão (dispersão absoluta dos dados em relação à média, numericamente)
- Coeficiente de variação (dispersão relativa dos dados em relação à média, em  porcentagem)
- Percentil - 95% e 99% (valores abaixo do qual uma certa porcentagem dos tempos se encontram)

> Todas as métricas levam em consideração a unidade de segundos para o tempo de resposta da requisição

In [ ]:
# Instalação de dependências
!pip install grpcio grpcio-tools pandas numpy

In [ ]:
# 2. Gere o arquivo .proto e compilar (O mesmo conteúdo usado no Gateway)
with open("summarizer.proto", "w") as f:
    f.write("""
    syntax = "proto3";
    service LLMGateway {
      rpc Summarize (SummarizeRequest) returns (SummarizeResponse) {}
    }
    message SummarizeRequest {
      string text = 1;
      string model_key = 2;
    }
    message SummarizeResponse {
      string summary = 1;
      int32 tokens_in = 2;
    }
    """)

# 3. Compile
!python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. summarizer.proto

import summarizer_pb2
import summarizer_pb2_grpc